In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../training_datasets")
OUTPUT_DIR = Path("../cleaned_datasets")

CHUNK_SIZE = 100_000

print("Cleaned datasets directory:")
print(OUTPUT_DIR.resolve())

Cleaned datasets directory:
/Users/sunny/Documents/Codes/Amazon ML 2026/cleaned_datasets


In [2]:
files = {
    "S1": OUTPUT_DIR / "cleaned_source1.tsv",
    "S2": OUTPUT_DIR / "cleaned_source2.tsv",
    "S3": OUTPUT_DIR / "cleaned_source3.tsv",
}

for source, path in files.items():
    print(f"{source}: {path}")
    print(f"  Exists: {path.exists()}")
    if path.exists():
        print(f"  Size:   {path.stat().st_size / (1024**2):.2f} MB")

S1: ../cleaned_datasets/cleaned_source1.tsv
  Exists: True
  Size:   367.05 MB
S2: ../cleaned_datasets/cleaned_source2.tsv
  Exists: True
  Size:   854.47 MB
S3: ../cleaned_datasets/cleaned_source3.tsv
  Exists: True
  Size:   882.62 MB


In [3]:
expected_columns = [
    "entity_id",
    "business_name",
    "business_address",
    "country",
    "clean_business_name",
    "clean_business_address",
    "zip_pin",
    "leading_number",
    "has_non_latin_script"
]

for source, path in files.items():
    header = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        nrows=0
    )

    actual_columns = header.columns.tolist()

    print(f"\n{source}")
    print("Columns:")
    for column in actual_columns:
        print(f"  {column}")

    print("Schema correct:", actual_columns == expected_columns)


S1
Columns:
  entity_id
  business_name
  business_address
  country
  clean_business_name
  clean_business_address
  zip_pin
  leading_number
  has_non_latin_script
Schema correct: True

S2
Columns:
  entity_id
  business_name
  business_address
  country
  clean_business_name
  clean_business_address
  zip_pin
  leading_number
  has_non_latin_script
Schema correct: True

S3
Columns:
  entity_id
  business_name
  business_address
  country
  clean_business_name
  clean_business_address
  zip_pin
  leading_number
  has_non_latin_script
Schema correct: True


In [4]:
expected_rows = {
    "S1": 2_206_821,
    "S2": 5_034_616,
    "S3": 5_285_603
}

for source, path in files.items():
    total_rows = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        chunksize=CHUNK_SIZE,
        keep_default_na=False,
        usecols=["entity_id"]
    ):
        total_rows += len(chunk)

    print(
        f"{source}: {total_rows:,} "
        f"(expected {expected_rows[source]:,}) "
        f"→ {'PASS' if total_rows == expected_rows[source] else 'FAIL'}"
    )

S1: 2,206,821 (expected 2,206,821) → PASS
S2: 5,034,616 (expected 5,034,616) → PASS
S3: 5,285,603 (expected 5,285,603) → PASS


In [5]:
expected_prefix = {
    "S1": "S1-",
    "S2": "S2-",
    "S3": "S3-"
}

for source, path in files.items():
    unique_ids = set()
    total_rows = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        chunksize=CHUNK_SIZE,
        keep_default_na=False,
        usecols=["entity_id"]
    ):
        ids = chunk["entity_id"]

        total_rows += len(ids)
        unique_ids.update(ids)

    bad_prefix = sum(
        not entity_id.startswith(expected_prefix[source])
        for entity_id in unique_ids
    )

    print(f"\n{source}")
    print(f"Rows:        {total_rows:,}")
    print(f"Unique IDs:  {len(unique_ids):,}")
    print(f"Duplicate IDs: {total_rows - len(unique_ids):,}")
    print(f"Bad prefixes:  {bad_prefix:,}")


S1
Rows:        2,206,821
Unique IDs:  2,206,821
Duplicate IDs: 0
Bad prefixes:  0

S2
Rows:        5,034,616
Unique IDs:  5,034,616
Duplicate IDs: 0
Bad prefixes:  0

S3
Rows:        5,285,603
Unique IDs:  5,285,603
Duplicate IDs: 0
Bad prefixes:  0


In [6]:
for source, path in files.items():
    print(f"\n{'=' * 80}")
    print(source)
    print("=" * 80)

    sample = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        nrows=10,
        keep_default_na=False
    )

    display(sample)


S1


,entity_id,business_name,business_address,country,clean_business_name,clean_business_address,zip_pin,leading_number,has_non_latin_script
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop,1795 westchester drive high point nc,,1795,False
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money,17560 ellis road tahlequah ok,,17560,False
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,b retail,1712 montebello avenue phoenix az,,1712,False
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,christ chapel,2100 cameron drive unit apartment g dundalk md,,2100,False
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,prabhav business center,797 lake town block a kolkata howrah west bengal,,797,False
5,S1-851869949,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",US,custom wealth services,oh columbus 5559 orville avenue,,,False
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India,consulting nyasa nursing,2505 tower 1 oakwood runwal greens mulund gore...,,2505,False
7,S1-27541239,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",US,nexus anchor rain,1111 church street unit 2007 nashville tn,,1111,False
8,S1-629417405,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",US,moore bitwise,337 oakland avenue michigan city in,,337,False
9,S1-22305073,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",US,dermatology green medicine,294 meadowcreek drive unit unit 2 village of p...,,294,False



S2


,entity_id,business_name,business_address,country,clean_business_name,clean_business_address,zip_pin,leading_number,has_non_latin_script
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,र म म र क ट ग प र इव ट ल म ट ड,kh no 570 13 new delhi west delhi delhi,,,True
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,holloway peak inc seafood,105 elm st morganton nc,,105,False
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,आद त य प र पर ट ज एलएलप,g 3 571 gulmohar colony bhopal madhya pradesh,,,True
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,summit,greensboro nc 19 1 2 stardust trail,,,False
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,delta tetlecommunication,914 pierpont ave cleveland oh,,914,False
5,S2-138046867,Lee and Lawson,"1702 Pine Avenue, CITY OF MENOMONIE, WI",US,lee and lawson,1702 pine avenue city of menomonie wi,,1702,False
6,S2-584977605,SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPOR...,"H.NO 204 C ROAD HOSHIARPUR, PUNJAB, Punjab",India,shivshakti vidyalaya vidyalaya overseas corpor...,h no 204 c road hoshiarpur punjab punjab,,,False
7,S2-277444929,Shree Infracon Private Ltd,"63/2275/7, ALHIND TOWER, FIRST FLOOR, JAFFERKH...",India,shree infracon private,63 2275 7 alhind tower first floor jafferkhan ...,,63,False
8,S2-721031885,Chavira Platinum Chimera LLC,"282 SAXONY DRIVE, FTT MITCHELL, KY",US,chavira platinum chimera,282 saxony drive ftt mitchell ky,,282,False
9,S2-508602797,FOUNDATION EXCEL AGENCY PRIVATE LIMITED,"HN 753 E-1, BHARAT NAGAR, 104/1/1 ERANDWANE, M...",India,foundation excel agency,hn 753 e 1 bharat nagar 104 1 1 erandwane maha...,,,False



S3


,entity_id,business_name,business_address,country,clean_business_name,clean_business_address,zip_pin,leading_number,has_non_latin_script
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US,wilfordhancock com,mack rd haltom city texas,,,False
1,S3-859268022,International South Consultants Private Ltd,,India,international south consultants private,,,,False
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US,llc moncada léarning center,5780 fawn ct fort worth texas,,5780,False
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US,moyna s coffee,1 ivanhoe ave po box 6009 cincinnati ohio,,1,False
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India,pvt efs print ventures,door no 183 41st cross 22nd main 9th block jay...,,,False
5,S3-578159284,LLC Hernandez Colonial Redwood,"2260- Housecreek Trail, Unit 407, Raleigh, Nor...",US,llc hernandez colonial redwood,2260 housecreek trail unit 407 raleigh north c...,,2260,False
6,S3-121412624,Classic Equity Partners Group,"6885 Catalpa Bluff Ln, PO Box 5799, Dickinson,...",US,classic equity partners group,6885 catalpa bluff ln po box 5799 dickinson texas,,6885,False
7,S3-249416830,Ectolumdrex dba X+ Madison Inc,"S03575 Cty Tk M, Town Of Buffalo, WI",US,ectolumdrex dba x madison,s03575 cty tk m town of buffalo wi,,,False
8,S3-107644605,Animal Hanisch Hospirlg,"##8 Willow Oak Lane, Fl. 0, Saint Louis, Missouri",US,animal hanisch hospirlg,8 willow oak lane fl 0 saint louis missouri,,8,False
9,S3-160217003,Gomez Optimal,"343 Hempstead 161, Hope, Arkansas",US,gomez optimal,343 hempstead 161 hope arkansas,,343,False


In [7]:
derived_columns = [
    "clean_business_name",
    "clean_business_address",
    "zip_pin",
    "leading_number",
    "has_non_latin_script"
]

for source, path in files.items():
    counts = {
        column: 0
        for column in derived_columns
    }

    total_rows = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        chunksize=CHUNK_SIZE,
        keep_default_na=False,
        usecols=derived_columns
    ):
        total_rows += len(chunk)

        for column in derived_columns:
            if column == "has_non_latin_script":
                continue

            counts[column] += (chunk[column] == "").sum()

    print(f"\n{source} — {total_rows:,} rows")

    for column in derived_columns:
        if column == "has_non_latin_script":
            continue

        missing = counts[column]
        percentage = missing / total_rows * 100

        print(
            f"{column:25} "
            f"{missing:>10,} "
            f"({percentage:6.2f}%)"
        )


S1 — 2,206,821 rows
clean_business_name                3 (  0.00%)
clean_business_address             0 (  0.00%)
zip_pin                    2,206,740 (100.00%)
leading_number               833,582 ( 37.77%)

S2 — 5,034,616 rows
clean_business_name               11 (  0.00%)
clean_business_address       168,967 (  3.36%)
zip_pin                    5,034,549 (100.00%)
leading_number             2,218,368 ( 44.06%)

S3 — 5,285,603 rows
clean_business_name               10 (  0.00%)
clean_business_address       175,916 (  3.33%)
zip_pin                    5,285,434 (100.00%)
leading_number             2,330,412 ( 44.09%)


In [8]:
for source, path in files.items():
    total = 0
    non_latin = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        chunksize=CHUNK_SIZE,
        keep_default_na=False,
        usecols=[
            "business_name",
            "clean_business_name",
            "has_non_latin_script"
        ]
    ):
        total += len(chunk)
        non_latin += (
            chunk["has_non_latin_script"]
            .str.lower()
            .eq("true")
        ).sum()

    print(
        f"{source}: "
        f"{non_latin:,} / {total:,} "
        f"({non_latin / total * 100:.2f}%) "
        f"flagged as non-Latin"
    )

S1: 0 / 2,206,821 (0.00%) flagged as non-Latin
S2: 474,345 / 5,034,616 (9.42%) flagged as non-Latin
S3: 278,524 / 5,285,603 (5.27%) flagged as non-Latin


In [9]:
test_addresses = [
    "17560 Ellis Road, Tahlequah, OK",
    "OH, Columbus, 5559 Orville Avenue",
    "##120 WOOD THRUSH LN, MOORESVILLE, NC",
    "123 Main Street, New York, NY 10001",
    "123 Main Street, New York, NY 10001-1234",
    "560034, Bangalore, Karnataka",
    "45A MG Road, Bengaluru 560001",
]

In [10]:
s1_sample = pd.read_csv(
    files["S1"],
    sep="\t",
    dtype=str,
    nrows=100_000,
    keep_default_na=False
)

s1_sample[
    [
        "business_name",
        "business_address",
        "clean_business_name",
        "clean_business_address",
        "zip_pin",
        "leading_number",
        "has_non_latin_script"
    ]
].head(20)

,business_name,business_address,clean_business_name,clean_business_address,zip_pin,leading_number,has_non_latin_script
0,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",orelee s barbershop,1795 westchester drive high point nc,,1795,False
1,Prime Money,"17560 Ellis Road, Tahlequah, OK",prime money,17560 ellis road tahlequah ok,,17560,False
2,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",b retail,1712 montebello avenue phoenix az,,1712,False
3,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",christ chapel,2100 cameron drive unit apartment g dundalk md,,2100,False
4,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",prabhav business center,797 lake town block a kolkata howrah west bengal,,797,False
5,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",custom wealth services,oh columbus 5559 orville avenue,,,False
6,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",consulting nyasa nursing,2505 tower 1 oakwood runwal greens mulund gore...,,2505,False
7,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",nexus anchor rain,1111 church street unit 2007 nashville tn,,1111,False
8,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",moore bitwise,337 oakland avenue michigan city in,,337,False
9,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",dermatology green medicine,294 meadowcreek drive unit unit 2 village of p...,,294,False


In [11]:
s1_sample = pd.read_csv(
    files["S1"],
    sep="\t",
    dtype=str,
    nrows=100_000,
    keep_default_na=False
)

print(s1_sample["zip_pin"].value_counts().head(20))

zip_pin
         99995
15308        1
13887        1
18105        1
42560        1
75182        1
Name: count, dtype: int64


In [12]:
non_empty = s1_sample[s1_sample["zip_pin"] != ""]

print(non_empty[
    [
        "business_address",
        "country",
        "zip_pin",
        "leading_number"
    ]
].head(20))

                                        business_address country zip_pin  \
30008  Portland, 400 Congress Street, ME, Unit UNIT 1...      US   15308   
33584  36234 Aspen Court, WI, City Of Independence, F...      US   13887   
69184  AZ, 12235 Thunderbird Road, El Mirage, Unit 18105      US   18105   
95500                   TX, Paris, 696 County Road 42560      US   42560   
98287                   224 Hearthstone Drive, TX, 75182      US   75182   

      leading_number  
30008                 
33584          36234  
69184                 
95500                 
98287            224  


In [13]:
s1_sample = pd.read_csv(
    files["S1"],
    sep="\t",
    dtype=str,
    nrows=100_000,
    keep_default_na=False
)

non_empty = s1_sample[s1_sample["zip_pin"] != ""]

for _, row in non_empty.iterrows():
    print(
        f"Address : {row['business_address']}\n"
        f"Country : {row['country']}\n"
        f"ZIP     : {row['zip_pin']}\n"
        f"Leading : {row['leading_number']}\n"
        f"{'-' * 100}"
    )

Address : Portland, 400 Congress Street, ME, Unit UNIT 15308
Country : US
ZIP     : 15308
Leading : 
----------------------------------------------------------------------------------------------------
Address : 36234 Aspen Court, WI, City Of Independence, Fl 13887
Country : US
ZIP     : 13887
Leading : 36234
----------------------------------------------------------------------------------------------------
Address : AZ, 12235 Thunderbird Road, El Mirage, Unit 18105
Country : US
ZIP     : 18105
Leading : 
----------------------------------------------------------------------------------------------------
Address : TX, Paris, 696 County Road 42560
Country : US
ZIP     : 42560
Leading : 
----------------------------------------------------------------------------------------------------
Address : 224 Hearthstone Drive, TX, 75182
Country : US
ZIP     : 75182
Leading : 224
----------------------------------------------------------------------------------------------------
